In [ ]:
import sqlite3
import re
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

Задание 1

In [ ]:
conn = sqlite3.connect('data/vacancies.db')

pd.read_sql('select * from vacancies', sqlite3.connect('data/test3.db')) \
    .rename(columns={"опыт'": 'опыт'}) \
    .to_sql('vacancies', conn, if_exists='replace', index=False)
pd.read_csv('data/test1.csv', sep='\t').to_sql('vacancies', conn, if_exists='append', index=False)
pd.read_csv('data/test2.csv', sep='\t').to_sql('vacancies', conn, if_exists='append', index=False)

df = pd.read_sql('select * from vacancies', conn)
df

Задание 2

In [ ]:
exp_years = {'не требуется': 0, '1–3 года': 2, '3–6 лет': 4.5, 'более 6 лет': 6}
df['exp'] = df['опыт'].map(exp_years)

to_rub = {'USD': 81.12, 'EUR': 91.13, 'KZT': 0.1808, 'бел.': 27.46, 'сум': 0.00678}

def parse_sal(s):
    s = str(s).replace('\xa0', ' ')
    nums = [int(n.replace(' ', '')) for n in re.findall(r'\d[\d ]*\d', s)]
    if not nums:
        return None
    rate = next((r for cur, r in to_rub.items() if cur in s), 1)
    return sum(nums) / len(nums) * rate

df['sal'] = df['зп'].map(parse_sal)
df[['опыт', 'exp', 'зп', 'sal']]

Задание 3

In [ ]:
df['region'] = df['регион'].str.split(',').str[0]
top_regions = df.loc[df['region'] != '', 'region'].value_counts().index[:10]

df[df['region'].isin(top_regions)].boxplot(column='sal', by='region', rot=90, figsize=(12, 6))
plt.show()

In [ ]:
skills = df.assign(skill=df['навыки'].map(ast.literal_eval)).explode('skill')
top_skills = skills['skill'].value_counts().index[:10]

skills[skills['skill'].isin(top_skills)].boxplot(column='sal', by='skill', rot=90, figsize=(12, 6))
plt.show()

Задание 4

In [ ]:
stats.spearmanr(df['exp'], df['sal'], nan_policy='omit')

In [ ]:
df['employment'] = df['занятость'].str.split(',').str[0]
table = pd.crosstab(df.loc[df['region'].isin(top_regions), 'region'], df['employment'])
chi2, p, dof, expected = stats.chi2_contingency(table)
table, p

Задание 5

In [ ]:
df['group'] = df['название'].map(lambda x: 'Seniors' if 'ведущ' in str(x).lower() or 'старш' in str(x).lower() else 'Others')
df['group'].value_counts()

Задание 6

In [ ]:
seniors = df.loc[df['group'] == 'Seniors', 'sal'].dropna()
others = df.loc[df['group'] == 'Others', 'sal'].dropna()
stats.ttest_ind(seniors, others, equal_var=False)

Задание 7

In [ ]:
n = 1_000_000
big = pd.DataFrame({
    'пол': np.random.choice(['М', 'Ж'], n),
    'возраст': np.random.normal(40, 12, n).astype(int),
    'доход': np.random.normal(80000, 20000, n).astype(int),
    'профессия': np.random.choice(['инженер', 'врач', 'учитель', 'программист'], n, p=[0.4, 0.3, 0.2, 0.1]),
})
big

Задание 8

In [ ]:
big.info(memory_usage='deep')

In [ ]:
big.memory_usage(deep=True)

Задание 9

In [ ]:
%%timeit -n 1 -r 3
big.to_csv('data/big.csv', index=False)

In [ ]:
%%timeit -n 1 -r 3
pd.read_csv('data/big.csv')

In [ ]:
big2 = pd.read_csv('data/big.csv')
big2

Задание 10

In [ ]:
%%timeit
big['возраст'].mean()
big['доход'].max()
big['пол'].mode()[0]
big['профессия'].value_counts().idxmin()

In [ ]:
big['возраст'].mean(), big['доход'].max(), big['пол'].mode()[0], big['профессия'].value_counts().idxmin()